In [7]:
#importing all the required libraries
import os
import glob
import pandas as pd
from collections import deque
from xgboost import XGBRegressor                                                    

# =====================================================================
# CONFIG
# =====================================================================
MODEL_DIR = r"C:\mit_supercloud_raw"                                                    #First few lines of the directory in which the model lies
MODEL_WINDOWS = [10, 30, 60, 120, 180, 240, 300, 360, 600]                              #Different model windows(10s,30s,...upto 600s)

FEATURES = [                                                                            #Input fetures list for XGBoost model
    "early_power_draw_W",
    "early_utilization_gpu_pct",
    "early_utilization_memory_pct",
    "early_memory_used_MiB",
    "early_memory_free_MiB",
    "early_temperature_gpu",
    "early_temperature_memory",
]

TIER_ORDER = ["Low", "Medium", "High", "Critical"]                                      #Different operating levels
TEMP_THRESHOLDS = {                                                                     #different temperature levels dictionary that control the cooling system working
    "Low": 35,
    "Medium": 45,
    "High": 71,
}

SAFETY_CEILING_C = 82.0                                                                 #Max_temp...after this the cooling system work in critical mode                                              
DEBOUNCE_PERSISTENCE = 3

TIER_TO_FAN_SPEED_PCT = {"Low": 40, "Medium": 60, "High": 80, "Critical": 100}          #Fan speed dictionary based on different level
TIER_TO_PUMP_SPEED_PCT = {"Low": 30, "Medium": 50, "High": 75, "Critical": 100}         #Pump speed dictionary based on different level
TIER_TO_VALVE_POS_PCT = {"Low": 25, "Medium": 45, "High": 70, "Critical": 100}          #Value position dictionary based on different level


# =====================================================================
# LAYER 1 — MODEL LOADING AND SELECTION (matches your 10s/30s/.../600s folders)
# =====================================================================

class ModelRegistry: 
    '''This class will do these things:
        *Creates the filepath of the XGboost models and load the models from the created path.
        *Select the suitable XGboost model based on elapsed seconds of the job.
        *Prediction of the output Temperature from the selected model.'''



    def __init__(self, model_dir, windows): 
        '''This function is called automatically when the object is created.
        This function will take first part address where the models located and different  
        time windows. '''
        
        self.models = {}                                                                #Empty dictionary to store models               
        for w in windows:                                                               #Looks for each window time inside the model window
            folder = os.path.join(model_dir, f"{w}s")                                   #Gives the final path of the folder
            
            if not os.path.isdir(folder):                                               # Check folder exists in the directory
                print(f"⚠️  Folder not found for {w}s window: {folder} — skipping.")
                continue
     
            json_files = glob.glob(os.path.join(folder, "*.json"))                      #Checks for .json file which has the inormation of pretrained model inside the folder
            if not json_files:                                                          # .json file is like the brain it consist all the information of the pretrained model
                print(f"⚠️  No .json model file found inside {folder} — skipping.")
                continue
            if len(json_files) > 1:                                                     #Checks if multiple json files are present inside the folder 
                print(f"⚠️  Multiple .json files found in {folder}, using the first: {json_files[0]}")

            model_path = json_files[0]                                                  #Chooses the first json file 
            model = XGBRegressor()                                                      #Creates an empty XGBoost regressor model
            model.load_model(model_path)                                                #The information of the pretrained model is fed to the created XGBoost model
            self.models[w] = model                                                      #Adds the loaded model to the dictionary
            print(f"✅ Loaded {w}s model from {model_path}")                      
                                                                                        #Till here all the models(like 10s model,30s model, etc) are loaded
        if not self.models:
            raise RuntimeError("No trained models were found. Check MODEL_DIR and folder structure.")  # If no model is found in the dictionary then we have to chek the provided directory

        ''' here the model dictionary will be in key-value pair like this
        self.models = {
                    10:  <XGBRegressor object>,
                    30:  <XGBRegressor object>,
                    60:  <XGBRegressor object>,
                    120: <XGBRegressor object>,
                    180: <XGBRegressor object>,
                    240: <XGBRegressor object>,
                    300: <XGBRegressor object>,
                    360: <XGBRegressor object>,
                    }'''
       
        self.available_windows = sorted(self.models.keys())                             #sort the model dictionay in ascending order based on keys and stores it in available models                           
        print(f"\n✅ Loaded models for windows: {self.available_windows} seconds")  



    def select_window(self, elapsed_seconds):
        ''' This function will select the model based on time elapsed by the job'''
        usable = []                                                                     #Creates an empty list to store the usable models
        for w in self.available_windows:
            if w<=elapsed_seconds:                                                      #check the each key (10s,30s....) in the available_windows is lessthan or equal to elapsed time by the job   
                usable.append(w)                                                        #All the usable model's window will be stored one after another in usable list
        if usable:
            return max(usable)   
            '''Returns the model with highest time window(eg if 150s has elaped 120s model will be returned).
            This ensures a correct model is selected based on the elaped time of job. '''
        else:   
            return self.available_windows[0]                                            #This will execute only if usable is empty. Then we will get the first model stored in available_windows 



    def predict(self, elapsed_seconds, feature_row: dict):
        ''' This function predicts the output using the selected model.
        It takes elapsed time of job and features required for prediction'''
        
        window = self.select_window(elapsed_seconds)                                   #Window (10s,30s,...) is selected using the function select_window()
        model = self.models[window]                                                    # Select the model based on the window 
        X = pd.DataFrame([feature_row], columns=FEATURES)                              #A dataframe of input features is created to fed to the model
        predicted_temp = float(model.predict(X)[0])                                    #Predicting Temperature based on model
        return predicted_temp, window                                                   


# =====================================================================
# LAYER 2 — DECISION LOGIC
'''In this layer, the GPU temperature is converted into different cooling tiers (Low, Medium, High, and Critical) and gives the actuation decision.'''
# =====================================================================

def temp_to_tier(temp_c: float) -> str:                                               #Takes float input and returns string output
    '''This function converts the temperature into different working levels of control(like low,high,medium,critical).
    If  GPU temperature is below 35'C it will be named as 'low' tier cooling job. If temperature is between 35'C to 45'C it will be 'Medium' tier
    cooling.Similarly if temperature is above 71'C it will be 'High' tier. And finally if temperature exceeds 81'C then it will be 'Critical' tier.'''
    
    if temp_c < TEMP_THRESHOLDS["Low"]: 
        return "Low"
    elif temp_c < TEMP_THRESHOLDS["Medium"]:
        return "Medium"
    elif temp_c < TEMP_THRESHOLDS["High"]:
        return "High"
    else:
        return "Critical"


def tier_rank(tier: str) -> int:                                                     #Takes string input and gives int output
    return TIER_ORDER.index(tier)                                                    # This function will return the index of the tier order( low=0, medium=1, high=2, critical=3)


def higher_tier(tier_a: str, tier_b: str) -> str:                                    #This function takes two strings and will return one string
    ''' In this function which tier rank is higher. For example if prediction layer gives rank as 'High' and sensor layer gives 'Critical' then the execution
    layer should consider 'Critical' or if prediction layer gives rank as 'High' and sensor layer gives 'Low' then the execution layer should consider 
    'High' sthis logic is executed in this function '''
    if tier_rank(tier_a) >= tier_rank(tier_b):
        return tier_a 
    else
        return tier_b


def apply_safety_ceiling(tier: str, current_sensor_temp: float) -> str:             #This function takes one string and one float and returns string      
    '''This function is for safety. If the actual temperature of the GPU exceed 82'C(Safety cieling ) then it will be considered as 'Critical' tier and
    the cooling systems will run at maximum capacity.'''    
    if current_sensor_temp >= SAFETY_CEILING_C:
        return "Critical"
    return tier


class TierDebouncer:
    '''The TierDebouncer class is used to prevent the Cooling tier from changing too quickly. Sometime there could be noise in the Sensory data so to 
    overcome this problem we have used this idea.A debouncer is a mechanism that prevents a system from reacting to temporary or noisy changes. 
    It changes the output only after the same input has been received consistently for a specified period or number of times.'''
    
    def __init__(self, persistence: int = DEBOUNCE_PERSISTENCE):                  #This function is called automatically when the object is created.It takes a user defined persistance value (here 3)
        self.persistence = persistence                                            #Persistance value set to coder defined persistance value
        self.history = deque(maxlen=persistence)                                  #A queue 'history' is created to which store the tier category(like 'low', 'high'etc) and has max length 3. 
                                                                                  #If the 4th item comes to queue then 1st item will be removed from queue
        self.active_tier = "Low"                                                  #Initially the working tier set to low
        

    def update(self, candidate_tier: str) -> str:                                 #This function is called whenever the prediction tier is recieved, it adds new tier to the history queue
        self.history.append(candidate_tier)
        if len(self.history) == self.persistence and len(set(self.history)) == 1: #Checks two conditions:The history queue has reached the required size (persistence) and all predictions in the history queue are identical
            self.active_tier = candidate_tier                                     #If both conditions are true, updates the active tier to the new stable prediction.
        return self.active_tier


def tier_to_actuator_command(tier: str, cooling_system_type: str = "air") -> dict: #This function takes 2 string values and gives a dictionary
    if cooling_system_type == "air":                                               #If the cooling system type is air this will return fan speed
        return {"fan_speed_pct": TIER_TO_FAN_SPEED_PCT[tier]}
    elif cooling_system_type == "liquid":                                          #If the cooling system type is liquid this will return pump speed and valve position
        return {
            "pump_speed_pct": TIER_TO_PUMP_SPEED_PCT[tier],
            "valve_position_pct": TIER_TO_VALVE_POS_PCT[tier],
        }
    else:                                                                          #If the cooling system doent matches then raise an error
        raise ValueError(f"Unknown cooling_system_type: {cooling_system_type}")


# =====================================================================
# FULL PIPELINE — one decision cycle for one job
# =====================================================================
class JobCoolingController:
    '''The JobCoolingController class manages the complete GPU cooling process for a single  job. It predicts the GPU temperature, 
    determines the cooling tier, applies safety checks and debouncing, generates the actuator commands, and returns the final cooling decision.'''
    
    def __init__(self, job_id, registry: ModelRegistry, cooling_system_type="air"):
        '''This function is called automaticlly when the object is created and it takes the job id,the ModelRegistry object that contains the 
        trained models and cooling system type.'''
        
        self.job_id = job_id                                                     #stores job id
        self.registry = registry                                                 #stores model registry 
        self.cooling_system_type = cooling_system_type                           #stores cooling system type
        self.debouncer = TierDebouncer()                                         #Create a TierDebouncer object.

    def step(self, elapsed_seconds: float, feature_row: dict, current_sensor_temp: float) -> dict:
        '''This is the main function that performs one complete cooling decision cycle. It takes the elapsed job time, input features, and the current GPU sensor temperature,
           and returns all the cooling decision details as a dictionary.'''
        predicted_temp, window_used = self.registry.predict(elapsed_seconds, feature_row)  # Predict the GPU temperature using the appropriate XGBoost model.
        predicted_tier = temp_to_tier(predicted_temp)                                    #working tier selection of predicted value
        sensor_tier = temp_to_tier(current_sensor_temp)                                  #working tier selection of sensor value

        combined_tier = higher_tier(predicted_tier, sensor_tier)                         #higher working tier got after comparing prediction and sensor value
        safe_tier = apply_safety_ceiling(combined_tier, current_sensor_temp)             # Apply the safety ceiling. If the actual GPU temperature exceeds the safety limit,the tier is forced to "Critical".                                                                              
        final_tier = self.debouncer.update(safe_tier)                                    # Stabilize the cooling tier using the debouncer to avoid rapid switching.

        command = tier_to_actuator_command(final_tier, self.cooling_system_type)         # Convert the final cooling tier into actuator commands.

        return {                                                                        # Return all the important information about the cooling decision,including temperatures, tiers, model used, and actuator commands.
            "job_id": self.job_id,
            "elapsed_seconds": elapsed_seconds,
            "model_window_used": window_used,
            "predicted_temp": round(predicted_temp, 2),
            "predicted_tier": predicted_tier,
            "current_sensor_temp": current_sensor_temp,
            "sensor_tier": sensor_tier,
            "combined_tier": combined_tier,
            "final_tier_after_debounce": final_tier,
            "actuator_command": command,
        }


# =====================================================================
# DEMO / TEST HARNESS
# =====================================================================
if __name__ == "__main__":                                                            # This ensures demo codde only executes when it is run directly not when the file is imported to another code
    registry = ModelRegistry(MODEL_DIR, MODEL_WINDOWS)                                # Create a model registry and load all trained XGBoost models

    controller1 = JobCoolingController(job_id="job_A", registry=registry, cooling_system_type="air") # Create a cooling controller for one GPU job with unique job id, registry and cooling system type-'air' denotes gpu system uses a air cooling system
                                                                                      # Pass the loaded model registry to the controller
    sample_features_1 = {                                                             # Sample GPU features used as input for prediction
        "early_power_draw_W": 150.0,                                                  # Current GPU power consumption in Watts
        "early_utilization_gpu_pct": 70.0,                                            # Current GPU utilization percentage
        "early_utilization_memory_pct": 55.0,                                         #Current GPU memory utilization percentage
        "early_memory_used_MiB": 12000.0,                                             # GPU memory currently in use (MiB)
        "early_memory_free_MiB": 20000.0,                                             # Available GPU memory (MiB)
        "early_temperature_gpu": 45.0,                                                # Current GPU core temperature 
        "early_temperature_memory": 42.0,                                             # Current GPU memory temperature
    }

    print("\n--- TEST 1: normal step, 120s elapsed ---")                              #Testing for 120s elapsed job time    
    result = controller1.step(elapsed_seconds=120, feature_row=sample_features_1, current_sensor_temp=46.0) # Perform one complete cooling decision cycle using the 120-second model
    for k, v in result.items():                                                       # Loop through all values returned by the controller
        print(f"  {k}: {v}")                                                          #printing values returned by the controller

    
    print("\n--- TEST 2: sensor spike, should force Critical ---")                   #Testing for high-temperature safety 
    result2 = controller1.step(elapsed_seconds=125, feature_row=sample_features_1, current_sensor_temp=84.0) # Simulate a dangerous temperature to test the safety override
    for k, v in result2.items():                                                     
        print(f"  {k}: {v}")                                                         #Display the results of the safety test

    
    print("\n--- TEST 3: debounce behavior over repeated noisy readings ---")       #tesing for debouncing
    controller3 = JobCoolingController(job_id="job_B", registry=registry, cooling_system_type="liquid")  # Create another controller for a different GPU job, use the same model registry but the cooling system type is liquid
    noisy_temps = [50.0, 72.0, 50.0, 72.0, 72.0, 72.0]                              # Simulated noisy sensor readings for debounce testing
    for i, t in enumerate(noisy_temps):                                             # Process each noisy sensor reading one by one
        r = controller3.step(elapsed_seconds=200 + i, feature_row=sample_features_1, current_sensor_temp=t) # Simulate increasing elapsed time for each reading,Simulate increasing elapsed time for each reading and Use the same input features for prediction and use the simulated noisy temperature
        print(f"  Check {i+1}: sensor={t}°C -> final_tier={r['final_tier_after_debounce']} " # Display the final cooling tier and actuator command after debouncing
              f"(command={r['actuator_command']})")

✅ Loaded 10s model from C:\mit_supercloud_raw\10s\model_a_10s_xgboost.json
✅ Loaded 30s model from C:\mit_supercloud_raw\30s\model_a_30s_xgboost.json
✅ Loaded 60s model from C:\mit_supercloud_raw\60s\model_a_FINAL(60s)_xgboost.json
✅ Loaded 120s model from C:\mit_supercloud_raw\120s\model_a_120s_xgboost.json
✅ Loaded 180s model from C:\mit_supercloud_raw\180s\model_a_180s_xgboost.json
✅ Loaded 240s model from C:\mit_supercloud_raw\240s\model_a_240s_xgboost.json
✅ Loaded 300s model from C:\mit_supercloud_raw\300s\model_a_300s_xgboost.json
✅ Loaded 360s model from C:\mit_supercloud_raw\360s\model_a_360s_xgboost.json
✅ Loaded 600s model from C:\mit_supercloud_raw\600s\model_a_600s_xgboost.json

✅ Loaded models for windows: [10, 30, 60, 120, 180, 240, 300, 360, 600] seconds

--- TEST 1: normal step, 120s elapsed ---
  job_id: job_A
  elapsed_seconds: 120
  model_window_used: 120
  predicted_temp: 69.89
  predicted_tier: High
  current_sensor_temp: 46.0
  sensor_tier: High
  combined_tier: 